# Work with Embeddings: Upload

This notebook uploads embeddings to a vector store for later use. It makes use of a vectore store API that has been developed for these experiments: the Vector database of the DH at Max Planck Society initiative (dhamps-vdb), developed by [Andreas Wagner](https://www.lhlt.mpg.de/wagner/en) of the [Max Planck Institute for Legal History and Legal Theory](https://www.lhlt.mpg.de/).

## 0. Preliminaries

In [ ]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/home/awagner/vcs/digicademy/svsal-bertopic/.venv/bin/python
3.11.14 (main, Dec 17 2025, 21:07:37) [Clang 21.1.4 ]
sys.version_info(major=3, minor=11, micro=14, releaselevel='final', serial=0)


## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [ ]:
import os
import logging
import dotenv
import polars as pl
import json
from tqdm.notebook import tqdm
from time import sleep
import numpy as np
import itertools
from functools import reduce
import pickle
import jsonlines
from typing import Any, Dict, List
import urllib
import requests

### 1.3 Configuration

### 1.4 API Keys Setup

In [ ]:
# Find the .env file and load it
dotenv.load_dotenv(dotenv.find_dotenv())

# Get API keys with fallbacks
api_keys = {
    "vdb": os.getenv("DHAMPS_VDB_API_KEY", "")
}
if api_keys["vdb"] == "":
    raise ValueError("API key for Vectordb (DHAMPS_VDB_API_KEY) not found in environment variables.")

print("API keys loaded successfully.")

In [ ]:
# VDB API configuration
API_USER = "sal"

# Provider-to-project mapping
# This maps embedding provider names to VDB project names
PROVIDER_PROJECT_MAPPING = {
    "openai": "sal-openai-small",
    "chat-ai-e5-mistral": "sal-e5-mistral",
    "chat-ai-multilingual-e5": "sal-multilingual-e5",
    "chat-ai-qwen3": "sal-qwen3",
    "cohere": "sal-cohere-v4",
    "gemini": "sal-gemini-001"
}

# Default configuration (can be overridden per provider)
HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {api_keys['vdb']}"}
BATCH_SIZE = None # Set to None to disable batching
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_REQUESTS = 0.2
VERBOSE = False

#### 1.3.2 File paths

In [ ]:
# Input files
file_path_in = './out-data'

# These files will be auto-detected from the most recent timestamp
# Priority order: parquet > pickle > csv (for docs)
# For embeddings: parquet > pickle

# We'll search for files with this pattern:
# YYYY-MM-DD_*_docs.{parquet,pkl,csv}
# YYYY-MM-DD_*_embeddings.{parquet,pkl}
# YYYY-MM-DD_*_processing_metadata.json
# YYYY-MM-DD_*_embedding_statistics.json

#### 1.3.3 Limits

In [ ]:
# Here we set the number of documents (paragraphs) to process
# Set it to a lower value until everything runs well, then increase it
# Set it to -1 to process all documents:
max_documents=5

## 2. Utility Functions

### 2.1 Logging Configuration

Configure structured logging for the embedding generation process.

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## 3. Read Input Data

### 3.1 Load data

In [ ]:
import glob
from pathlib import Path

def find_latest_files(directory: str, pattern: str) -> str:
    """Find the most recent file matching the pattern."""
    files = glob.glob(os.path.join(directory, pattern))
    if not files:
        return None
    # Sort by modification time, most recent first
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def load_docs_data(directory: str) -> pl.DataFrame:
    """Load docs data from parquet, pickle, or CSV, in that order."""
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_docs.parquet")
    if parquet_file:
        print(f"Loading docs from parquet: {parquet_file}")
        return pl.read_parquet(parquet_file)
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_docs.pkl")
    if pickle_file:
        print(f"Loading docs from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            df = pickle.load(f)
            # If it's a pandas DataFrame, convert to polars
            if hasattr(df, 'to_dict'):  # pandas DataFrame check
                return pl.from_pandas(df)
            return df
    
    # Try CSV
    csv_file = find_latest_files(directory, "*_docs.csv")
    if csv_file:
        print(f"Loading docs from CSV: {csv_file}")
        return pl.read_csv(csv_file)
    
    raise FileNotFoundError(f"No docs files found in {directory}")

def load_embeddings_data(directory: str) -> dict:
    """Load embeddings data from parquet or pickle, in that order.
    Returns a nested dict: {provider_name: {doc_id: [embedding_vector]}}
    """
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_embeddings.parquet")
    if parquet_file:
        print(f"Loading embeddings from parquet: {parquet_file}")
        # Parquet stores nested dict as serialized format - need to reconstruct
        df_embeddings = pl.read_parquet(parquet_file)
        # Convert back to nested dict structure
        embeddings_dict = {}
        for row in df_embeddings.iter_rows(named=True):
            provider = row['provider']
            doc_id = row['doc_id']
            embedding = row['embedding']
            if provider not in embeddings_dict:
                embeddings_dict[provider] = {}
            embeddings_dict[provider][doc_id] = embedding
        return embeddings_dict
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_embeddings.pkl")
    if pickle_file:
        print(f"Loading embeddings from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            return pickle.load(f)
    
    raise FileNotFoundError(f"No embeddings files found in {directory}")

def load_metadata(directory: str) -> dict:
    """Load processing metadata JSON."""
    metadata_file = find_latest_files(directory, "*_processing_metadata.json")
    if metadata_file:
        print(f"Loading metadata from: {metadata_file}")
        with open(metadata_file, "r") as f:
            return json.load(f)
    return {}

# Load all data
print("="*80)
print("Loading data files...")
print("="*80)

docs = load_docs_data(file_path_in)
embeddings_dict = load_embeddings_data(file_path_in)
metadata = load_metadata(directory=file_path_in)

print(f"\nLoaded {docs.height} documents")
print(f"Loaded embeddings for {len(embeddings_dict)} providers:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  - {provider}: {len(provider_embeddings)} documents")

if metadata:
    print(f"\nProcessing metadata:")
    print(f"  Processing date: {metadata.get('processing_date', 'N/A')}")
    if 'providers' in metadata:
        print(f"  Providers in metadata: {list(metadata['providers'].keys())}")

### 3.2 Give some information about the data

In [ ]:
print("="*80)
print("Data Overview")
print("="*80)
print(f"\nShape of docs dataframe: {docs.shape}")
print(f"Number of available documents: {docs.height}")
print(f"\nDocument length statistics:")
if "len_content" in docs.columns:
    print(docs["len_content"].describe())
else:
    print("  (len_content column not available)")

print("\nEmbeddings by provider:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  {provider}: {len(provider_embeddings)} embeddings")
    # Show embedding dimension
    if provider_embeddings:
        sample_embedding = next(iter(provider_embeddings.values()))
        print(f"    - Dimension: {len(sample_embedding)}")

print("\nFirst 3 rows of docs dataframe:")
docs.head(3)

## 4. Uploading to Vector Store

Define functions to build uploadable data either from dataframe plus embeddings, or from dataframe including embeddings.

In [ ]:
def build_json_object(
    doc_id: str, 
    doc_row: dict, 
    embedding: list, 
    provider_name: str,
    api_user: str,
    api_project: str,
    api_project_id: int,
    llm_service_handle: str
) -> dict:
    """Build a JSON object for a single document embedding for VDB upload."""
    
    # Build the json object
    json_object = {
        "text_id": urllib.parse.quote(doc_id, safe=''),
        "user_handle": api_user,
        "project_handle": api_project,
        "project_id": api_project_id,
        "llm_service_handle": llm_service_handle,
        "text": doc_row["content"],
        "vector": embedding,
        "vector_dim": len(embedding),
        "metadata": {
            "url": doc_id,
            "xmlid": doc_row["xmlid"],
            "wid": doc_row["wid"],
            "author": doc_row["author-name"],
            "author_id": doc_row["author-id"],
            "year": int(doc_row["year"]) if isinstance(doc_row["year"], (np.int64, int)) else doc_row["year"],
            "lang": doc_row["lang"],
        }
    }
    return json_object

def build_json_objects_for_provider(
    provider_name: str,
    docs_df: pl.DataFrame,
    embeddings: dict,
    api_user: str,
    api_project: str,
    api_project_id: int,
    llm_service_handle: str
) -> list:
    """Build JSON objects for all documents with embeddings from a specific provider.
    
    Args:
        provider_name: Name of the embedding provider
        docs_df: Polars DataFrame with document data
        embeddings: Dict mapping doc_id to embedding vector
        api_user: VDB API user handle
        api_project: VDB API project handle
        api_project_id: VDB API project ID
        llm_service_handle: LLM service identifier
        
    Returns:
        List of JSON objects ready for VDB upload
    """
    json_objects = []
    
    # Create a lookup dict from the dataframe for faster access
    docs_dict = {row["url"]: row for row in docs_df.iter_rows(named=True)}
    
    # Iterate over the embeddings for this provider
    for doc_id, embedding in tqdm(embeddings.items(), desc=f"Building payloads for {provider_name}"):
        if doc_id not in docs_dict:
            print(f"Warning: Document {doc_id} not found in docs dataframe")
            continue
        
        if not embedding or len(embedding) == 0:
            print(f"Warning: Empty embedding for document {doc_id}")
            continue
        
        doc_row = docs_dict[doc_id]
        json_obj = build_json_object(
            doc_id=doc_id,
            doc_row=doc_row,
            embedding=embedding,
            provider_name=provider_name,
            api_user=api_user,
            api_project=api_project,
            api_project_id=api_project_id,
            llm_service_handle=llm_service_handle
        )
        json_objects.append(json_obj)
    
    print(f"Built {len(json_objects)} JSON objects for {provider_name}")
    return json_objects

def build_all_json_objects(
    docs_df: pl.DataFrame,
    embeddings_dict: dict,
    provider_project_mapping: dict,
    api_user: str
) -> dict:
    """Build JSON objects for all providers.
    
    Args:
        docs_df: Polars DataFrame with document data
        embeddings_dict: Nested dict {provider: {doc_id: embedding}}
        provider_project_mapping: Dict mapping provider names to VDB project names
        api_user: VDB API user handle
        
    Returns:
        Dict mapping provider names to lists of JSON objects
    """
    all_json_objects = {}
    
    for provider_name, provider_embeddings in embeddings_dict.items():
        if provider_name not in provider_project_mapping:
            print(f"Warning: No project mapping for provider '{provider_name}', skipping")
            continue
        
        project_name = provider_project_mapping[provider_name]
        # Note: Project IDs should ideally be fetched from VDB API or configuration
        # For now, we'll use a simple hash-based approach or sequential numbering
        project_id = hash(project_name) % 10000  # Temporary solution
        
        json_objects = build_json_objects_for_provider(
            provider_name=provider_name,
            docs_df=docs_df,
            embeddings=provider_embeddings,
            api_user=api_user,
            api_project=project_name,
            api_project_id=project_id,
            llm_service_handle=provider_name
        )
        
        all_json_objects[provider_name] = json_objects
    
    return all_json_objects

Next, define a function to upload data to our vector store.

In [ ]:
class JsonListApiSender:
    def __init__(
        self,
        api_endpoint: str,
        headers: Dict[str, str] = None,
        retry_attempts: int = 3,
        delay_between_requests: float = 0.1,
        verbose: bool = True
    ):
        """
        Initialize the JSON List API Sender.
        
        Args:
            api_endpoint: URL to send POST requests to
            headers: Optional headers for the request
            retry_attempts: Number of times to retry a failed request
            delay_between_requests: Delay between each request
            verbose: Whether to print detailed logging
        """
        self.api_endpoint = API_ENDPOINT
        self.headers = HEADERS or {"Content-Type": "application/json"}
        self.retry_attempts = RETRY_ATTEMPTS
        self.delay_between_requests = DELAY_BETWEEN_REQUESTS
        self.verbose = VERBOSE
        # self.logger = logging.getLogger(__name__)

    def send_single_item(self, item: Dict[str, Any]) -> bool:
        """
        Send a single JSON item to the API with retry logic.
        
        Args:
            item: JSON object to send
        
        Returns:
            Boolean indicating success or failure
        """
        for attempt in range(self.retry_attempts):
            try:
                response = requests.post(
                    self.api_endpoint,
                    json={"embeddings": [item]},
                    headers=self.headers
                )
                response.raise_for_status()
                
                if self.verbose:
                    print(f"Successfully sent item with text_id: {item.get('text_id')}")
                
                return True
            
            except requests.exceptions.RequestException as e:
                # Log error only for the last attempt
                if attempt == self.retry_attempts - 1:
                    error = e.read() if hasattr(e, 'read') else str(e)
                    try:
                        response
                        try:
                            response.content
                            print(f"Failed to send item. Error: {str(error)}\nResponse: {response.content}\nItem: {item}")
                        except AttributeError:
                            print(f"Failed to send item. Error: {str(error)}\nItem: {item}")

                    except NameError:
                        print(f"Failed to send item. Error: {str(error)}\nItem: {item}")
                
                # Exponential backoff
                sleep(2 ** attempt)
        
        return False

    def send_list(
        self, 
        json_list: List[Dict[str, Any]], 
        num_items: int = BATCH_SIZE
    ) -> Dict[str, int]:
        """
        Send a list of JSON objects to the API.
        
        Args:
            json_list: List of JSON objects to send
            num_items: Number of items to send (None = send all)
        
        Returns:
            Dictionary with counts of successful and failed items
        """
        # Determine number of items to send
        if num_items is None:
            num_items = len(json_list)
        else:
            num_items = min(num_items, len(json_list))
        
        # Slice the list to the desired number of items
        items_to_send = json_list[:num_items]
        
        # Tracking variables
        success_count = 0
        fail_count = 0
        failed_items = set()
        
        # Log start of sending process
        print(f"Starting to send {num_items} items to {self.api_endpoint}")
        
        # Progress bar
        for item in tqdm(items_to_send, desc="Sending items", total=num_items):
            if self.send_single_item(item):
                success_count += 1
            else:
                fail_count += 1
                failed_items.add(item.get('text_id'))
            
            # Delay between requests
            sleep(self.delay_between_requests)
        
        # Log overall results
        self.logger.info(
            f"Sending complete. "
            f"Sent: {num_items}, "
            f"Successful: {success_count}, "
            f"Failed: {fail_count}"
        )
        
        return {
            "total_sent": num_items,
            "successful": success_count,
            "failed": fail_count,
            "failed_ids": list(failed_items)
        }

# Example usage:
# sender = JsonListApiSender(api_endpoint="https://your-api-endpoint.com", verbose=True)
# results = sender.send_list(your_json_list, num_items=10)

Initialize sender

In [ ]:
sender = JsonListApiSender(api_endpoint=API_ENDPOINT, verbose=VERBOSE)

For those records that have not been uploaded yet, do it now.

In [ ]:
# for embeddings entries that have not yet been uploaded, build json objects and upload them
unuploaded_dict = {k: v for k, v in embeddings_dict.items() if k not in uploaded_ids}
upload_objects = build_json_objects_df_dict(docs, unuploaded_dict)
sender.send_list(upload_objects)

  0%|          | 0/71988 [00:00<?, ?it/s]

Done. Number of json objects: 71988
Starting to send 71988 items to https://c100-188.cloud.gwdg.de/vdb-api/v1/embeddings/sal/sal-openai-large


Sending items:   0%|          | 0/71988 [00:00<?, ?it/s]